# Get Line of stations from Renfe data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('..')

In [3]:
import datetime

from src.robin.scraping.renfe.entities import RenfeScraper

scraper = RenfeScraper(stations_csv_path='../data/renfe/renfe_stations.csv')

for station_id, station_name in scraper.available_stations.items():
    print(f'{station_id}: {station_name}')

?: Estaciones de Origen
31412: A Coruña
94707: Abrantes
60911: Alicante / Alacant
60600: Albacete
06008: Alcantarilla-Los Romanos
60400: Alcázar de San Juan
55020: Algeciras
56312: Almería
99003: Altet Bus
99115: Aguadulce Bus
87912: Aix En Provence
99114: Andorra-Bus
ANTEQ: Antequera (TODAS)
87814: Avignon
10400: Avila
37606: Badajoz
BARCE: Barcelona (TODAS)
87078: Beziers
65318: Benicassim
BILBA: Bilbao (TODAS)
54400: Bobadilla
11014: Burgos Rosa Manzano
35400: Cáceres
51405: Cádiz
70600: Calatayud
50417: Campus Rabanales
61307: Cartagena
65300: Castellón /Castelló
37200: Ciudad Real
50500: Córdoba
CUENC: Cuenca (TODAS)
92201: Denia-Bus
60905: Elda-Petrer
03410: Elche AV/Elx AV
94428: Entroncamento
92157: Estepona Bus
21010: Ferrol
79309: Figueres
79333: Figueres Bus
04307: Figueres Vilafant
69110: Gandía
GIJON: Gijón
79300: Girona
05000: Granada
GUADA: Guadalajara (TODAS)
43019: Huelva
74200: Huesca
IRUN-: Irun-Hendaya (TODAS)
80100: Pamplona/Iruña
99103: Jaca-Bus
03100: Jaén
64100:

In [4]:
scraper.stations_df

,stop_id,stop_name,renfe_id,stop_lat,stop_lon
0,00000,Unknown,00000,0.000000,0.000000
1,31412,A Corunya,31412,43.352761,-8.409755
2,60911,AlicanteAlacant,60911,38.344450,-0.495053
3,60600,Albacete-Los Llanos,60600,38.999384,-1.848450
4,60400,Alcazar de San Juan,60400,39.395628,-3.205744
...,...,...,...,...,...
92,13200,Bilbao-Abando Indalecio Prieto,BILBA,43.259609,-2.929150
93,66100,Cuenca,CUENC,40.067340,-2.136471
94,15410,GijonXixon,GIJON,43.535175,-5.698318
95,70200,Guadalajara,GUADA,40.644103,-3.182230


In [5]:
import pandas as pd 

from itertools import combinations

df_trips, df_stops = pd.DataFrame(), pd.DataFrame()
        
origin_id = 'MADRI'
destination_id = 'BARCE'
stations = ['MADRI', 'GUADA', '70600', 'ZARAG', '78400', 'TARRA', 'BARCE']

# Generate all possible trips from stations
combos = list(combinations(stations, 2))

for trip in combos:
    origin, destination = trip
    print(trip)

('MADRI', 'GUADA')
('MADRI', '70600')
('MADRI', 'ZARAG')
('MADRI', '78400')
('MADRI', 'TARRA')
('MADRI', 'BARCE')
('GUADA', '70600')
('GUADA', 'ZARAG')
('GUADA', '78400')
('GUADA', 'TARRA')
('GUADA', 'BARCE')
('70600', 'ZARAG')
('70600', '78400')
('70600', 'TARRA')
('70600', 'BARCE')
('ZARAG', '78400')
('ZARAG', 'TARRA')
('ZARAG', 'BARCE')
('78400', 'TARRA')
('78400', 'BARCE')
('TARRA', 'BARCE')


In [6]:
# 15, 17, 19, 20, 21
date = datetime.date(day=25, month=6, year=2024)

for trip in combos: 
    origin_id, destination_id = trip
    buffer_df_trips, buffer_df_stops = scraper.scrape_trips(origin_id=origin_id, 
                                                            destination_id=destination_id, 
                                                            init_date=date,
                                                            range_days=1)
    
    df_trips = pd.concat([df_trips, buffer_df_trips], ignore_index=True)
    df_stops = pd.concat([df_stops, buffer_df_stops], ignore_index=True)
print(f"######## ROWS OF DF_TRIPS: {df_trips.shape[0]} ############")
        
print(df_stops.head())

Date:  2024-06-25
Search url:  https://horarios.renfe.com/HIRRenfeWeb/buscar.do?O=MADRI&D=GUADA&AF=2024&MF=06&DF=25&SF=2&ID=s
##################################################
UNKNOWN STATION: tafalla
##################################################
##################################################
UNKNOWN STATION: pamplonairuna
##################################################
##################################################
UNKNOWN STATION: perpignan
##################################################
##################################################
UNKNOWN STATION: narbonne
##################################################
##################################################
UNKNOWN STATION: montpellier saint roch
##################################################
##################################################
UNKNOWN STATION: nimes
##################################################
##################################################
UNKNOWN STATION: avignon tgv
##########

In [10]:
from datetime import datetime

df_stops['date'] = df_stops['service_id'].apply(lambda sid: "/".join(sid.split("_")[-1].split("-")[:-1]))
df_stops['stop'] = df_stops['stop_id'].apply(lambda sid: scraper.stations_df[scraper.stations_df['stop_id'] == sid]['stop_name'].values[0])

df_stops['renfe_id'] = df_stops['service_id'].apply(lambda sid: sid.split("_")[0])
mask = df_stops['stop_id'].isin(['60000', '70600', '04040', '71801', '70200', '78400', '71500'])
df_stops_filtered = df_stops[mask]

df_stops_new = pd.DataFrame()
for group in df_stops_filtered.groupby('renfe_id'):
    service_sids = {sid: datetime.strptime(sid.split("_")[-1], "%d-%m-%Y-%H.%M") for sid in set(group[1]['service_id'])}
    sorted_sids = dict(sorted(service_sids.items(), key=lambda t: t[1]))
    print(sorted_sids)
    min_sid = tuple(sorted_sids.keys())[0]
    mask = df_stops['service_id'].isin([min_sid])
    group_filtered = group[1][mask]
    df_stops_new = pd.concat([df_stops_new, group_filtered], ignore_index=True)

df_stops_new.drop_duplicates()

{'00437_25-06-2024-19.52': datetime.datetime(2024, 6, 25, 19, 52), '00437_25-06-2024-20.58': datetime.datetime(2024, 6, 25, 20, 58), '00437_25-06-2024-21.32': datetime.datetime(2024, 6, 25, 21, 32)}
{'00533_25-06-2024-11.09': datetime.datetime(2024, 6, 25, 11, 9), '00533_25-06-2024-12.00': datetime.datetime(2024, 6, 25, 12, 0), '00533_25-06-2024-12.33': datetime.datetime(2024, 6, 25, 12, 33)}
{'00605_25-06-2024-10.35': datetime.datetime(2024, 6, 25, 10, 35), '00605_25-06-2024-11.00': datetime.datetime(2024, 6, 25, 11, 0)}
{'00609_25-06-2024-14.50': datetime.datetime(2024, 6, 25, 14, 50), '00609_25-06-2024-15.16': datetime.datetime(2024, 6, 25, 15, 16)}
{'00625_25-06-2024-19.24': datetime.datetime(2024, 6, 25, 19, 24), '00625_25-06-2024-20.14': datetime.datetime(2024, 6, 25, 20, 14), '00625_25-06-2024-20.54': datetime.datetime(2024, 6, 25, 20, 54)}
{'00631_25-06-2024-19.24': datetime.datetime(2024, 6, 25, 19, 24), '00631_25-06-2024-20.14': datetime.datetime(2024, 6, 25, 20, 14), '00631_

/var/folders/dc/fy76phnd5ljbdtj9v4jgnyjh0000gp/T/ipykernel_17100/3084748888.py:17: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  group_filtered = group[1][mask]
/var/folders/dc/fy76phnd5ljbdtj9v4jgnyjh0000gp/T/ipykernel_17100/3084748888.py:17: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  group_filtered = group[1][mask]
/var/folders/dc/fy76phnd5ljbdtj9v4jgnyjh0000gp/T/ipykernel_17100/3084748888.py:17: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  group_filtered = group[1][mask]
/var/folders/dc/fy76phnd5ljbdtj9v4jgnyjh0000gp/T/ipykernel_17100/3084748888.py:17: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  group_filtered = group[1][mask]
/var/folders/dc/fy76phnd5ljbdtj9v4jgnyjh0000gp/T/ipykernel_17100/3084748888.py:17: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  group_filtered = group[1][mask]
/var/folders/dc/fy76phnd5ljbdtj9v4j

,service_id,stop_id,arrival,departure,date,stop,renfe_id
0,00437_25-06-2024-19.52,04040,269,272,25/06/2024,Zaragoza-Delicias,00437
1,00437_25-06-2024-19.52,78400,336,338,25/06/2024,Lleida,00437
2,00437_25-06-2024-19.52,71500,370,372,25/06/2024,Tarragona,00437
3,00437_25-06-2024-19.52,71801,420,420,25/06/2024,Barcelona-Sants,00437
12,00533_25-06-2024-11.09,04040,244,247,25/06/2024,Zaragoza-Delicias,00533
...,...,...,...,...,...,...,...
495,34943_25-06-2024-15.26,04040,25,25,25/06/2024,Zaragoza-Delicias,34943
496,34963_25-06-2024-07.26,70600,0,0,25/06/2024,Calatayud,34963
497,34963_25-06-2024-07.26,04040,25,25,25/06/2024,Zaragoza-Delicias,34963
498,34993_25-06-2024-20.06,70600,0,0,25/06/2024,Calatayud,34993


In [11]:
df_stops_new

,service_id,stop_id,arrival,departure,date,stop,renfe_id
0,00437_25-06-2024-19.52,04040,269,272,25/06/2024,Zaragoza-Delicias,00437
1,00437_25-06-2024-19.52,78400,336,338,25/06/2024,Lleida,00437
2,00437_25-06-2024-19.52,71500,370,372,25/06/2024,Tarragona,00437
3,00437_25-06-2024-19.52,71801,420,420,25/06/2024,Barcelona-Sants,00437
4,00437_25-06-2024-19.52,04040,269,272,25/06/2024,Zaragoza-Delicias,00437
...,...,...,...,...,...,...,...
495,34943_25-06-2024-15.26,04040,25,25,25/06/2024,Zaragoza-Delicias,34943
496,34963_25-06-2024-07.26,70600,0,0,25/06/2024,Calatayud,34963
497,34963_25-06-2024-07.26,04040,25,25,25/06/2024,Zaragoza-Delicias,34963
498,34993_25-06-2024-20.06,70600,0,0,25/06/2024,Calatayud,34993


In [12]:
import numpy as np
stations = ['Madrid-Puerta de Atocha', 'Guadalajara', 'Calatayud', 'Zaragoza-Delicias',
            'Lleida', 'Tarragona', 'Barcelona-Sants', 'Girona', 'Figueres Vilafant']

summary_trips = {}
for i, group in enumerate(df_stops_new.groupby('date')):
    trips = {}
    grouped = group[1].groupby("service_id")
    
    # Iteramos sobre los grupos generados en el paso anterior
    for name, group in grouped:
        # Para cade tren, generamos un diccionario con su ruta
        service_trip = {}
        for index, row in group.iterrows():
            service_trip[row['stop']] = (row['arrival'], row['departure'])
        trips[name] = service_trip
    
    for t in trips:
        trip_filtered = tuple(filter(lambda s: s in stations, tuple(trips[t].keys())))
        if trip_filtered in summary_trips:
            summary_trips[trip_filtered] += 1
        else:
            summary_trips[trip_filtered] = 1
        
total_trips = sum(summary_trips.values())
day_perc = {k: np.round(summary_trips[k] / total_trips, 2) for k in summary_trips}

for trip in day_perc:
    print("New trip: ")
    for stop in trip:
        print(f"\t{stop}")
    print(f"Percentage: {day_perc[trip]}")

New trip: 
	Zaragoza-Delicias
	Lleida
	Tarragona
	Barcelona-Sants
Percentage: 0.19
New trip: 
	Madrid-Puerta de Atocha
	Guadalajara
	Calatayud
Percentage: 0.06
New trip: 
	Madrid-Puerta de Atocha
	Guadalajara
	Calatayud
	Zaragoza-Delicias
Percentage: 0.02
New trip: 
	Madrid-Puerta de Atocha
	Calatayud
	Zaragoza-Delicias
	Barcelona-Sants
Percentage: 0.04
New trip: 
	Madrid-Puerta de Atocha
	Barcelona-Sants
Percentage: 0.25
New trip: 
	Madrid-Puerta de Atocha
	Guadalajara
	Zaragoza-Delicias
	Lleida
	Tarragona
	Barcelona-Sants
Percentage: 0.04
New trip: 
	Madrid-Puerta de Atocha
	Zaragoza-Delicias
	Barcelona-Sants
Percentage: 0.06
New trip: 
	Madrid-Puerta de Atocha
	Zaragoza-Delicias
	Lleida
	Tarragona
	Barcelona-Sants
Percentage: 0.02
New trip: 
	Madrid-Puerta de Atocha
	Calatayud
	Zaragoza-Delicias
	Lleida
	Tarragona
	Barcelona-Sants
Percentage: 0.04
New trip: 
	Madrid-Puerta de Atocha
	Zaragoza-Delicias
	Lleida
	Barcelona-Sants
Percentage: 0.02
New trip: 
	Madrid-Puerta de Atocha
	Zar

In [13]:
day_perc

{('Zaragoza-Delicias', 'Lleida', 'Tarragona', 'Barcelona-Sants'): 0.19,
 ('Madrid-Puerta de Atocha', 'Guadalajara', 'Calatayud'): 0.06,
 ('Madrid-Puerta de Atocha',
  'Guadalajara',
  'Calatayud',
  'Zaragoza-Delicias'): 0.02,
 ('Madrid-Puerta de Atocha',
  'Calatayud',
  'Zaragoza-Delicias',
  'Barcelona-Sants'): 0.04,
 ('Madrid-Puerta de Atocha', 'Barcelona-Sants'): 0.25,
 ('Madrid-Puerta de Atocha',
  'Guadalajara',
  'Zaragoza-Delicias',
  'Lleida',
  'Tarragona',
  'Barcelona-Sants'): 0.04,
 ('Madrid-Puerta de Atocha', 'Zaragoza-Delicias', 'Barcelona-Sants'): 0.06,
 ('Madrid-Puerta de Atocha',
  'Zaragoza-Delicias',
  'Lleida',
  'Tarragona',
  'Barcelona-Sants'): 0.02,
 ('Madrid-Puerta de Atocha',
  'Calatayud',
  'Zaragoza-Delicias',
  'Lleida',
  'Tarragona',
  'Barcelona-Sants'): 0.04,
 ('Madrid-Puerta de Atocha',
  'Zaragoza-Delicias',
  'Lleida',
  'Barcelona-Sants'): 0.02,
 ('Madrid-Puerta de Atocha',
  'Zaragoza-Delicias',
  'Tarragona',
  'Barcelona-Sants'): 0.02,
 ('Madr

In [14]:
for i, k  in enumerate(day_perc, start=1):
    print(f"      '{i}': {day_perc[k]}")

      '1': 0.19
      '2': 0.06
      '3': 0.02
      '4': 0.04
      '5': 0.25
      '6': 0.04
      '7': 0.06
      '8': 0.02
      '9': 0.04
      '10': 0.02
      '11': 0.02
      '12': 0.04
      '13': 0.02
      '14': 0.06
      '15': 0.02
      '16': 0.08


In [15]:
sum(day_perc.values())

0.9800000000000003

In [16]:
# Travel times analysis
summary_sts = {}
for i, group in enumerate(df_stops_new.groupby('service_id')):
    service_trip = {}
    for index, row in group[1].iterrows():
        service_trip[row['stop']] = (row['arrival'], row['departure'])
    
    stops = tuple(service_trip.keys())
    for j in range(1, len(stops) - 1):
        stop = stops[j]
        stop_time = service_trip[stop][1] - service_trip[stop][0]
        if stop in summary_sts:
            summary_sts[stop].append(stop_time)
        else:
            summary_sts[stop] = [stop_time]

statistics_sts = {}
for t in summary_sts:
    statistics_sts[t] = {
        'mean': np.mean(summary_sts[t]),
        'max': np.max(summary_sts[t]),
        'min': np.min(summary_sts[t]),
        'std': np.std(summary_sts[t])
    }

statistics_sts

{'Lleida': {'mean': 2.0, 'max': 2, 'min': 2, 'std': 0.0},
 'Tarragona': {'mean': 2.0, 'max': 2, 'min': 2, 'std': 0.0},
 'Guadalajara': {'mean': 1.1111111111111112,
  'max': 2,
  'min': 1,
  'std': 0.3142696805273545},
 'Calatayud': {'mean': 1.0, 'max': 1, 'min': 1, 'std': 0.0},
 'Zaragoza-Delicias': {'mean': 1.1333333333333333,
  'max': 3,
  'min': 1,
  'std': 0.49888765156985887}}

In [17]:
# Travel times analysis
summary_tts = {}
for i, group in enumerate(df_stops_new.groupby('service_id')):
    service_trip = {}
    for index, row in group[1].iterrows():
        service_trip[row['stop']] = (row['arrival'], row['departure'])
    
    stops = tuple(service_trip.keys())
    for j in range(len(stops) - 1):
        trip = (stops[j], stops[j + 1])
        travel_time = service_trip[trip[1]][0] - service_trip[trip[0]][1]
    
        if trip in summary_tts:
            summary_tts[trip].append(travel_time)
        else:
            summary_tts[trip] = [travel_time]

statistics_tts = {}
for t in summary_tts:
    print(f"From {t[0]} to {t[1]}")
    print(f"\tMean travel time: {np.mean(summary_tts[t])}")
    print(f"\tMax travel time: {np.max(summary_tts[t])}")
    print(f"\tMin travel time: {np.min(summary_tts[t])}")
    print(f"\tStd travel time: {np.std(summary_tts[t])}")
    print()
    
    statistics_tts[t] = {
        'mean': np.mean(summary_tts[t]),
        'max': np.max(summary_tts[t]),
        'min': np.min(summary_tts[t]),
        'std': np.std(summary_tts[t])
    }

statistics_tts

From Zaragoza-Delicias to Lleida
	Mean travel time: 44.705882352941174
	Max travel time: 64
	Min travel time: 40
	Std travel time: 5.5811100085319625

From Lleida to Tarragona
	Mean travel time: 28.736842105263158
	Max travel time: 38
	Min travel time: 24
	Std travel time: 4.586503142954599

From Tarragona to Barcelona-Sants
	Mean travel time: 37.666666666666664
	Max travel time: 48
	Min travel time: 34
	Std travel time: 3.212080372198105

From Madrid-Puerta de Atocha to Guadalajara
	Mean travel time: 23.22222222222222
	Max travel time: 25
	Min travel time: 22
	Std travel time: 0.7856742013183862

From Guadalajara to Calatayud
	Mean travel time: 41.833333333333336
	Max travel time: 53
	Min travel time: 35
	Std travel time: 6.465721580423608

From Calatayud to Zaragoza-Delicias
	Mean travel time: 24.727272727272727
	Max travel time: 25
	Min travel time: 24
	Std travel time: 0.4453617714151233

From Madrid-Puerta de Atocha to Calatayud
	Mean travel time: 55.0
	Max travel time: 55
	Min tr

{('Zaragoza-Delicias', 'Lleida'): {'mean': 44.705882352941174,
  'max': 64,
  'min': 40,
  'std': 5.5811100085319625},
 ('Lleida', 'Tarragona'): {'mean': 28.736842105263158,
  'max': 38,
  'min': 24,
  'std': 4.586503142954599},
 ('Tarragona', 'Barcelona-Sants'): {'mean': 37.666666666666664,
  'max': 48,
  'min': 34,
  'std': 3.212080372198105},
 ('Madrid-Puerta de Atocha', 'Guadalajara'): {'mean': 23.22222222222222,
  'max': 25,
  'min': 22,
  'std': 0.7856742013183862},
 ('Guadalajara', 'Calatayud'): {'mean': 41.833333333333336,
  'max': 53,
  'min': 35,
  'std': 6.465721580423608},
 ('Calatayud', 'Zaragoza-Delicias'): {'mean': 24.727272727272727,
  'max': 25,
  'min': 24,
  'std': 0.4453617714151233},
 ('Madrid-Puerta de Atocha', 'Calatayud'): {'mean': 55.0,
  'max': 55,
  'min': 55,
  'std': 0.0},
 ('Zaragoza-Delicias', 'Barcelona-Sants'): {'mean': 88.8,
  'max': 90,
  'min': 88,
  'std': 0.7483314773547882},
 ('Madrid-Puerta de Atocha', 'Barcelona-Sants'): {'mean': 149.91666666666

In [18]:
day_perc

{('Zaragoza-Delicias', 'Lleida', 'Tarragona', 'Barcelona-Sants'): 0.19,
 ('Madrid-Puerta de Atocha', 'Guadalajara', 'Calatayud'): 0.06,
 ('Madrid-Puerta de Atocha',
  'Guadalajara',
  'Calatayud',
  'Zaragoza-Delicias'): 0.02,
 ('Madrid-Puerta de Atocha',
  'Calatayud',
  'Zaragoza-Delicias',
  'Barcelona-Sants'): 0.04,
 ('Madrid-Puerta de Atocha', 'Barcelona-Sants'): 0.25,
 ('Madrid-Puerta de Atocha',
  'Guadalajara',
  'Zaragoza-Delicias',
  'Lleida',
  'Tarragona',
  'Barcelona-Sants'): 0.04,
 ('Madrid-Puerta de Atocha', 'Zaragoza-Delicias', 'Barcelona-Sants'): 0.06,
 ('Madrid-Puerta de Atocha',
  'Zaragoza-Delicias',
  'Lleida',
  'Tarragona',
  'Barcelona-Sants'): 0.02,
 ('Madrid-Puerta de Atocha',
  'Calatayud',
  'Zaragoza-Delicias',
  'Lleida',
  'Tarragona',
  'Barcelona-Sants'): 0.04,
 ('Madrid-Puerta de Atocha',
  'Zaragoza-Delicias',
  'Lleida',
  'Barcelona-Sants'): 0.02,
 ('Madrid-Puerta de Atocha',
  'Zaragoza-Delicias',
  'Tarragona',
  'Barcelona-Sants'): 0.02,
 ('Madr

In [19]:
for i, trip in enumerate(day_perc, start=1):
    trip_formatted = [stop.split("-")[0] for stop in trip]
    elapsed_time = 0
    print(f"- id: '{i}'")
    line_name = " - ".join([sta[:3].upper() for sta in trip_formatted])
    print(f"  name: '{line_name}'")
    print(f"  corridor: '1'")
    print(f"  stops:")
    for j in range(len(trip) - 1):
        origin = trip[j]
        destination = trip[j + 1]
        travel_time = statistics_tts[(origin, destination)]['mean']
        print(f"  - station: '{trip_formatted[j]}'")
        arrival_time = elapsed_time
        stop_time = round(statistics_sts.get(origin, {'mean': 0})['mean'])
        departure_time = arrival_time + stop_time
        print(f"    arrival_time: {arrival_time}")
        print(f"    departure_time: {departure_time}")
        elapsed_time += round(travel_time + stop_time)
        
    print(f"  - station: '{trip_formatted[-1]}'")
    arrival_time = round(elapsed_time)
    departure_time = arrival_time
    print(f"    arrival_time: {arrival_time}")
    print(f"    departure_time: {departure_time}")
    #print(f"Percentage: {day_perc[trip]}")

- id: '1'
  name: 'ZAR - LLE - TAR - BAR'
  corridor: '1'
  stops:
  - station: 'Zaragoza'
    arrival_time: 0
    departure_time: 1
  - station: 'Lleida'
    arrival_time: 46
    departure_time: 48
  - station: 'Tarragona'
    arrival_time: 77
    departure_time: 79
  - station: 'Barcelona'
    arrival_time: 117
    departure_time: 117
- id: '2'
  name: 'MAD - GUA - CAL'
  corridor: '1'
  stops:
  - station: 'Madrid'
    arrival_time: 0
    departure_time: 0
  - station: 'Guadalajara'
    arrival_time: 23
    departure_time: 24
  - station: 'Calatayud'
    arrival_time: 66
    departure_time: 66
- id: '3'
  name: 'MAD - GUA - CAL - ZAR'
  corridor: '1'
  stops:
  - station: 'Madrid'
    arrival_time: 0
    departure_time: 0
  - station: 'Guadalajara'
    arrival_time: 23
    departure_time: 24
  - station: 'Calatayud'
    arrival_time: 66
    departure_time: 67
  - station: 'Zaragoza'
    arrival_time: 92
    departure_time: 92
- id: '4'
  name: 'MAD - CAL - ZAR - BAR'
  corridor: '1'

In [26]:
import pandas as pd 

trips = ['60000', '65000', '60600']

"""
         '71801',
         '74200',  # HUESCA
         '15100',  # LEÓN
         '11014',  # BURGOS
         '31412',  # A CORUÑA
         '23004',  # PONTEVEDRA
         '65300',  # CASTELLÓN
         '60911']  # ALICANTE
"""

master = trips[0]
destinations = trips[1:]

df_trips, df_stops = pd.DataFrame(), pd.DataFrame()

for master in trips:
    for destination in trips[trips.index(master)+1:]:
        for i in range(2):
            trip = None
            if i == 0:
                print(f'From {master} to {destination}')
                trip = (master, destination)
            else:
                print(f'From {destination} to {master}')
                trip = (destination, master)
            
            origin_id = scraper.get_renfe_station_id(trip[0])
            destination_id = scraper.get_renfe_station_id(trip[1])
        
            # 15, 17, 19, 20, 21
            date = datetime.date(day=21, month=6, year=2024)
        
            buffer_df_trips, buffer_df_stops = scraper.scrape_trips(origin_id=origin_id, 
                                                                    destination_id=destination_id, 
                                                                    init_date=date)
            
            df_trips = pd.concat([df_trips, buffer_df_trips], ignore_index=True)
            df_stops = pd.concat([df_stops, buffer_df_stops], ignore_index=True)
            print(f"######## ROWS OF DF_TRIPS: {df_trips.shape[0]} ############")
        
print(df_stops.head())

From 60000 to 65000
Date:  2024-06-21
Search url:  https://horarios.renfe.com/HIRRenfeWeb/buscar.do?O=MADRI&D=VALEN&AF=2024&MF=06&DF=21&SF=5&ID=s
######## ROWS OF DF_TRIPS: 12 ############
From 65000 to 60000
Date:  2024-06-21
Search url:  https://horarios.renfe.com/HIRRenfeWeb/buscar.do?O=VALEN&D=MADRI&AF=2024&MF=06&DF=21&SF=5&ID=s
######## ROWS OF DF_TRIPS: 23 ############
From 60000 to 60600
Date:  2024-06-21
Search url:  https://horarios.renfe.com/HIRRenfeWeb/buscar.do?O=MADRI&D=60600&AF=2024&MF=06&DF=21&SF=5&ID=s
##################################################
UNKNOWN STATION: torrelavegatanos
##################################################
##################################################
UNKNOWN STATION: reinosa
##################################################
######## ROWS OF DF_TRIPS: 35 ############
From 60600 to 60000
Date:  2024-06-21
Search url:  https://horarios.renfe.com/HIRRenfeWeb/buscar.do?O=60600&D=MADRI&AF=2024&MF=06&DF=21&SF=5&ID=s
########################

In [29]:
len(set(list(map(lambda sid: sid.split("_")[0], set(df_stops['service_id'])))))

48

In [30]:
df_stops.head()

,service_id,stop_id,arrival,departure
0,05064_21-06-2024-06.30,60000,0,0
1,05064_21-06-2024-06.30,70101,63,65
2,05064_21-06-2024-06.30,03213,98,99
3,05064_21-06-2024-06.30,65000,124,124
4,05070_21-06-2024-07.30,60000,0,0


In [31]:
# Len of dataframe
print(len(df_stops))

227


In [32]:
# Remove rows with stop_id value '00000'
df_stops_new = df_stops[df_stops['stop_id'] != '00000']
print(len(df_stops_new))

225


In [33]:
print(set(df_stops_new['stop_id']))

{'92102', '70101', '60400', '60911', '11014', '65000', '14100', '60000', '62002', '12100', '61200', '14223', '60600', '03213', '15100', '03410', '10600', '03309', '65300'}


In [34]:
for adif_id in set(df_stops_new['stop_id']):
    name = scraper.stations_df[scraper.stations_df['stop_id'] == adif_id]['stop_name'].values[0]
    print(name)

Toledo
San Fernando Henares
Alcazar de San Juan
AlicanteAlacant
Burgos-Rosa de Lima
Valencia-Estacio del Nord
Palencia
Madrid-Puerta de Atocha
Orihuela-Miguel Hernandez
Segovia
Murcia
Santander
Albacete-Los Llanos
Requena Utiel
Leon
ELX AV
Valladolid
Villena Av
Castello


In [10]:
# Group by service_id. 
grouped = df_stops_new.groupby("service_id")

# If first row of group has arrival and departure not equal to 0, then remove the group
for name, group in grouped:
    if group.iloc[0]['arrival'] != 0 and group.iloc[0]['departure'] != 0:
        df_stops_new = df_stops_new[df_stops_new['service_id'] != name]

print(len(df_stops_new))

192


In [11]:
# Print number of groups
print(len(grouped))

36


In [12]:
for name, group in grouped:
    if group.iloc[-1]['arrival'] != group.iloc[-1]['departure']:
        df_stops_new = df_stops_new[df_stops_new['service_id'] != name]

print(len(df_stops_new))

192


In [13]:
# If len of group is less than 2, then remove the group
for name, group in grouped:
    if len(group) < 2:
        df_stops_new = df_stops_new[df_stops_new['service_id'] != name]

print(len(df_stops_new))

192


In [53]:
# Save dataframe to csv
df_stops_new.to_csv('stops_HSR_Spain_20_April_24.csv', index=False)

In [10]:
# Save dataframe to csv
df_trips.to_csv('trips_HSR_Spain_15_April_24.csv', index=False)

In [11]:
import pandas as pd

dtypes = {'stop_id': str}
df_stops = pd.read_csv('stops_HSR_Spain_15_April_24.csv', dtype=dtypes)
df_stops.head()

,service_id,stop_id,arrival,departure
0,06301_15-04-2024-06.15,60000,0,0
1,06301_15-04-2024-06.15,71801,150,150
2,03063_15-04-2024-06.30,60000,0,0
3,03063_15-04-2024-06.30,70600,55,56
4,03063_15-04-2024-06.30,04040,81,82


In [11]:
# Len of dataframe
print(len(df_stops))

6241


In [12]:
print(set(df_stops['stop_id']))

{'04040', '71500', '79300', '04307', '70600', '70200', '60000', '00000', '71801', '78400'}


In [13]:
print(set(scraper.stations_df['stop_id']))

{'64100', '14100', '79309', '15211', '50500', '37704', '70101', '03410', '65304', '23021', '81100', '92102', '03213', '50417', '51400', '11014', '65318', '37606', '54413', '06006', '78301', '30100', '62002', '15410', '61307', '15100', '31200', '55020', '20309', '22300', '10400', '78200', '81202', '12100', '60000', '14223', '11208', '51300', '71801', '04040', '03309', '60905', '05000', '69110', '60902', '04307', '02002', '35206', '40100', '78400', '20300', '74200', '35400', '60911', '60400', '15009', '37200', '10600', '23004', '51003', '79315', '50300', '31400', '08223', '70600', '66100', '00000', '02003', '43019', '31412', '37300', '70200', '30200', '11200', '21010', '61200', '67200', '71400', '37500', '20200', '56312', '80100', '50102', '79300', '71500', '54400', '22100', '11511', '82100', '40008', '13200', '65000', '51405', '10500', '60600', '03100', '65300'}


In [14]:
# Get set of stations which are NOT in the column renfe_id of scraper.stations_df
print(set(df_stops['stop_id']) - set(scraper.stations_df['stop_id']))

set()


In [8]:
# remove rows in wich stop_id is not in the column renfe_id of scraper.stations_df
df_stops_clean = df_stops[df_stops['stop_id'].isin(scraper.stations_df['renfe_id'])]
df_stops_clean.reset_index(drop=True, inplace=True)
print(df_stops_clean)

                   service_id stop_id  arrival  departure
0      03063_01-03-2024-06.30   70600       55         56
1      03073_01-03-2024-07.30   78400      125        127
2      03093_01-03-2024-09.30   78400      119        121
3      03093_01-03-2024-09.30   79300      233        235
4      03093_01-03-2024-09.30   04307      250        250
...                       ...     ...      ...        ...
13673  05903_31-03-2024-20.01   03309       81         83
13674  05903_31-03-2024-20.01   60600      115        117
13675  05217_31-03-2024-21.04   00000        0          0
13676  05217_31-03-2024-21.04   03309       19         21
13677  05217_31-03-2024-21.04   60600       53         55

[13678 rows x 4 columns]


In [10]:
# save dataframe to csv
df_stops_clean.to_csv('stops_HSR_Spain_March_24_clean.csv', index=False)

In [15]:
result_dict = {}

grouped = df_stops.groupby("service_id")

for name, group in grouped:
    sub_dict = {}
    for index, row in group.iterrows():
        sub_dict[row['stop_id']] = (row['arrival'], row['departure'])
    result_dict[name] = sub_dict

print(result_dict)

{'03062_01-06-2024-05.50': {'71801': (0, 0), '71500': (33, 35), '78400': (60, 62), '04040': (103, 106), '70600': (130, 131), '70200': (169, 170), '60000': (200, 200)}, '03062_03-06-2024-05.50': {'71801': (0, 0), '71500': (33, 35), '78400': (60, 62), '04040': (103, 106), '70600': (130, 131), '70200': (169, 170), '60000': (200, 200)}, '03062_04-06-2024-05.50': {'71801': (0, 0), '71500': (33, 35), '78400': (60, 62), '04040': (103, 106), '70600': (130, 131), '70200': (169, 170), '60000': (200, 200)}, '03062_05-06-2024-05.50': {'71801': (0, 0), '71500': (33, 35), '78400': (60, 62), '04040': (103, 106), '70600': (130, 131), '70200': (169, 170), '60000': (200, 200)}, '03062_06-06-2024-05.50': {'71801': (0, 0), '71500': (33, 35), '78400': (60, 62), '04040': (103, 106), '70600': (130, 131), '70200': (169, 170), '60000': (200, 200)}, '03062_07-06-2024-05.50': {'71801': (0, 0), '71500': (33, 35), '78400': (60, 62), '04040': (103, 106), '70600': (130, 131), '70200': (169, 170), '60000': (200, 200)

In [16]:
trips = list(set(tuple(service.keys())for service in result_dict.values()))
trips

[('04307', '79300', '71801', '04040', '60000'),
 ('60000', '04040', '78400', '71500', '71801', '79300', '04307'),
 ('60000', '70200', '04040', '71500', '71801', '79300', '04307', '00000'),
 ('60000', '04040', '71500', '71801'),
 ('04307', '79300', '71801', '71500', '78400', '04040', '70600', '60000'),
 ('71801', '71500', '78400', '04040', '70200', '60000'),
 ('60000', '71801', '79300', '04307'),
 ('60000', '70600', '04040', '71801'),
 ('60000', '04040', '71801', '79300', '04307'),
 ('71801', '71500', '78400', '04040', '70600', '70200', '60000'),
 ('60000', '70200', '04040', '78400', '71500', '71801'),
 ('00000', '04307', '79300', '71801', '71500', '04040', '60000'),
 ('71801', '71500', '78400', '04040', '70600', '60000'),
 ('60000', '04040', '78400', '71801', '79300', '04307'),
 ('71801', '60000'),
 ('60000',
  '70200',
  '70600',
  '04040',
  '78400',
  '71500',
  '71801',
  '79300',
  '04307'),
 ('04307', '79300', '71801', '60000'),
 ('60000', '70600', '04040', '71801', '79300', '043

In [17]:
# Initialize line with max length trip
line_stations = list(trips.pop(trips.index(max(trips, key=len))))

# Complete corridor with other stops that are not in the initial defined corridor
for trip in trips:
    for i, station in enumerate(trip):
        if station not in line_stations:
            # If station is the last one, append it to the end of the corridor
            if i == len(trip) - 1:
                line_stations.append(station)
            else:
                # If station is not the last one, insert it in the corridor before the next station
                index = line_stations.index(trip[i + 1])
                line_stations.insert(index, station)

print(line_stations)

['60000', '70200', '70600', '04040', '78400', '71500', '71801', '79300', '04307', '00000']


In [18]:
mapped_names = scraper.stations_df.set_index('stop_id')['stop_name'].to_dict()
line_stations_names = list(map(mapped_names.get, line_stations))

print(line_stations_names)

['Madrid-Puerta de Atocha', 'Guadalajara', 'Calatayud', 'Zaragoza-Delicias', 'Lleida', 'Tarragona', 'Barcelona-Sants', 'Girona', 'Figueres Vilafant', 'Unknown']


In [ ]:
# ['Madrid-Puerta de Atocha', 'Guadalajara', 'Calatayud', 'Zaragoza-Delicias', 'Tardienta', 'Huesca']

['Madrid-Puerta de Atocha', 'Guadalajara', 'Calatayud', 'Zaragoza-Delicias', [['Tarragona', 'Lleida', 'Barcelona-Sants', 'Girona', 'Figueres Vilafant'], ['Tardienta', 'Huesca']]]

In [ ]:
['60000', '70200', '70600', '04040'], [['71500', '78400', '71801', '79300', '04307'], ['78200', '74200']]

In [19]:
df_stations = scraper.stations_df.copy()

print(df_stations.head())

  stop_id            stop_name renfe_id   stop_lat  stop_lon
0   00000              Unknown    00000   0.000000  0.000000
1   31412            A Corunya    31412  43.352761 -8.409755
2   60911     Alicante/alacant    60911  38.344450 -0.495053
3   60600  Albacete-Los Llanos    60600  38.999384 -1.848450
4   60400  Alcazar de San Juan    60400  39.395628 -3.205744


In [22]:
estaciones_noreste = ['60000', '70200', '70600', '04040', '71500', '78400', '71801', '79300', '04307', '78200', '74200']

df_stations = df_stations[df_stations['stop_id'].isin(estaciones_noreste)]
df_stations.drop(columns=['renfe_id'], inplace=True)
df_stations.reset_index(drop=True, inplace=True)
print(df_stations)

   stop_id                stop_name   stop_lat  stop_lon
0    70600                Calatayud  41.346692 -1.638680
1    04307        Figueres Vilafant  42.264771  2.943547
2    79300                   Girona  41.979303  2.817006
3    74200                   Huesca  42.133594 -0.409745
4    78400                   Lleida  41.620696  0.632669
5    60000  Madrid-Puerta de Atocha  40.406442 -3.690886
6    71500                Tarragona  41.111624  1.253214
7    04040        Zaragoza-Delicias  41.658649 -0.911615
8    71801          Barcelona-Sants  41.379220  2.140624
9    70200              Guadalajara  40.644103 -3.182230
10   78200                Tardienta  41.975751 -0.538314


In [23]:
# save dataframe to csv
df_stations.to_csv('estaciones_corredor_noreste.csv', index=False)